# Código Artículo de EDO's
**Descripción:** En este archivo Jupyter se encuentra los 3 distintos métodos para detectar periodos en la gráfica de producción cientifica vs las citaciones. Los métodos se encuentran en el siguiente orden:
1. Metódo Derivadas
2. Método Regresión Lineal por Tramos
3. Detección de Rupturas Estructurales con PELT

## Librerias Utilizadas

In [2]:
# Importar librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
import os # Libreria para crear rutas de archivos


# Librerías Primer Método
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d


# Librerías Segundo Método
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import itertools


# Librerías Tercer Método
from sklearn.cluster import KMeans
from scipy.ndimage import gaussian_filter1d

## Creación de rutas de archivos

In [3]:
# Carpeta donde está el notebook
base_path = os.getcwd()

# Nombres de las carpetas
folders = [
    "1_Método_Derivadas",
    "2_Método_Regresión-Lineal-Tramos",
    "3_Método_Detección-Rupturas-Estructurales-PELT"
]

# Verificar y crear carpetas
for folder in folders:
    folder_path = os.path.join(base_path, folder)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Carpeta creada: {folder}")
    else:
        print(f"Carpeta ya existe: {folder}")


Carpeta creada: 1_Método_Derivadas
Carpeta creada: 2_Método_Regresión-Lineal-Tramos
Carpeta creada: 3_Método_Detección-Rupturas-Estructurales-PELT


## Datos Utilizados
Cada pareja de arrays (`years` & `values`) representa una gráfica de publicaciones a través del tiempo, por lo tanto, para usar alguna se debe comentar o ejecutar la respectiva celda, y luego elegir que algoritmo se quiere utilizar para separar los periodos, o ejecutar los 3 para encontrar las diferencias. 

In [4]:
# --- Datos --- 1
# years = np.array([
#     2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013,
#     2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024
# ])
# values = np.array([
#     1, 0, 0, 2, 0, 1, 3, 0, 0, 1, 2, 2, 5, 4, 16, 31, 50, 50, 85, 127, 141, 142
# ])


# --- Datos --- 2 (Profesor Sebastián Robledo)
years = np.array(
    [2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 
         2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
)
values = np.array(
    [2, 0, 2, 1, 3, 5, 7, 5, 9, 5, 
    6, 6, 3, 3, 12, 13, 19, 11, 10, 23, 19]
          )

## 1. Metódo Derivadas

In [ ]:
# --- Suavizado ---
smoothed = gaussian_filter1d(values, sigma=2)

# --- Derivadas ---
d1 = np.gradient(smoothed)
d2 = np.gradient(d1)

# --- Detectar puntos de cambio (picos en la 2ª derivada) ---
peaks_idx, _ = find_peaks(np.abs(d2), height=np.std(d2))
change_years = years[peaks_idx]

# --- Añadir inicio y fin como bordes de periodos ---
period_limits = np.sort(np.concatenate(([years[0]], change_years, [years[-1]])))

# --- Calcular crecimiento por periodo ---
periods = []
for i in range(len(period_limits)-1):
    start = period_limits[i]
    end = period_limits[i+1]
    start_val = values[np.where(years == start)][0]
    end_val = values[np.where(years == end)][0]
    growth = ((end_val - start_val) / start_val * 100) if start_val > 0 else np.nan
    periods.append((start, end, start_val, end_val, growth))

# --- Graficar ---
plt.figure(figsize=(12,5))
plt.plot(years, values, "o-", label="Publicaciones (datos)")
plt.plot(years, smoothed, "-", label="Suavizado")

# Mostrar el número exacto en cada punto
for x, y in zip(years, values):
    plt.text(x, y + 2, str(y), ha="center", va="bottom", fontsize=8, color="black")

# Marcar años de cambio
for y in change_years:
    plt.axvline(x=y, color="red", linestyle="--", alpha=0.7)
    plt.text(y, max(values)*0.9, str(y), rotation=90, 
             color="red", fontsize=10, ha="center", va="bottom")

# Mostrar info de periodos en la gráfica
for (start, end, s_val, e_val, growth) in periods:
    label = f"{start}-{end}\n{growth:.1f}%"
    plt.text((start+end)/2, max(values)*0.6, label,
             ha="center", va="center", fontsize=9,
             bbox=dict(facecolor="white", alpha=0.6, edgecolor="gray"))

plt.xticks(years, rotation=45)
plt.xlabel("Año")
plt.ylabel("Número de publicaciones")
plt.title("Detección de etapas y crecimiento porcentual")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

# --- Guardar ---
plt.savefig("cambio_etapas_con_crecimiento.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Mostrar resultados en consola ---
print("Periodos detectados con crecimiento:")
for (start, end, s_val, e_val, growth) in periods:
    print(f"{start}-{end}: {s_val} → {e_val}, Crecimiento = {growth:.1f}%")


: 

## 2. Método Regresión Lineal por Tramos

In [ ]:
# ==========================
# Función para ajuste por tramos (mínimo 3 años por tramo)
# ==========================
def piecewise_linear_fit(years, values, n_breaks=2, min_len=3):
    """
    Ajuste por regresión lineal por tramos que minimiza el error cuadrático total.
    Explora todas las combinaciones posibles de puntos de cambio.
    """
    n = len(years)
    best_sse = np.inf
    best_breaks = None

    # Todas las combinaciones posibles de puntos de cambio
    possible_breaks = range(min_len, n - min_len + 1)
    for breaks in itertools.combinations(possible_breaks, n_breaks):
        breaks = sorted(list(breaks))
        segments = np.split(values, breaks)
        year_segments = np.split(years, breaks)

        # Calcula el SSE total
        sse = 0
        for x, y in zip(year_segments, segments):
            model = LinearRegression().fit(x.reshape(-1, 1), y)
            y_pred = model.predict(x.reshape(-1, 1))
            sse += np.sum((y - y_pred) ** 2)

        if sse < best_sse:
            best_sse = sse
            best_breaks = breaks

    return best_breaks

# ==========================
# Ajuste con 2 puntos de cambio (3 periodos)
# ==========================
break_indices = piecewise_linear_fit(years, values, n_breaks=2, min_len=3)
break_years = [years[i - 1] for i in break_indices]
segments = np.split(values, break_indices)
year_segments = np.split(years, break_indices)

# ==========================
# Gráfica
# ==========================
plt.figure(figsize=(11, 6))
plt.plot(years, values, 'o-', color='black', label='Datos originales')

for i, (x, y) in enumerate(zip(year_segments, segments)):
    model = LinearRegression().fit(x.reshape(-1, 1), y)
    y_pred = model.predict(x.reshape(-1, 1))
    plt.plot(x, y_pred, linewidth=3)

    # Calcular crecimiento total del periodo (porcentaje total)
    start, end = y[0], y[-1]
    if start > 0:
        growth = ((end - start) / start) * 100
        growth_text = f"+{growth:.1f}%"
    else:
        growth_text = "N/A"

    # Posición del texto en el centro del tramo
    mid_x = x[len(x)//2]
    mid_y = y_pred[len(x)//2]
    plt.text(mid_x, mid_y + 1.5, growth_text, fontsize=10, color='blue', ha='center', fontweight='bold')

# ==========================
# Añadir líneas verticales y etiquetas de los años
# ==========================
for by in break_years:
    plt.axvline(by, color='red', linestyle='--', linewidth=1.2)
    plt.text(by + 0.2, plt.ylim()[1] * 0.95, str(by),
             color='red', fontsize=10, rotation=90, va='top', ha='left', fontweight='bold')

# ==========================
# Formato final
# ==========================
plt.title('Periodización por regresión lineal por tramos (2004–2024)')
plt.xlabel('Año')
plt.ylabel('Publicaciones')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 3. Detección de Rupturas Estructurales con PELT

In [ ]:



# --- 1. Suavizado Gaussiano ---
smoothed = gaussian_filter1d(values, sigma=2)

# --- 2. Calcular las pendientes suavizadas ---
slopes = np.diff(smoothed) / np.diff(years)

# --- 3. Agrupar pendientes con K-means ---
k = 4  # número máximo de periodos
model = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = model.fit_predict(slopes.reshape(-1, 1))

# --- 4. Detectar los puntos de cambio (cuando cambia el cluster) ---
change_points = [0]
for i in range(1, len(clusters)):
    if clusters[i] != clusters[i - 1]:
        change_points.append(i)
change_points.append(len(values) - 1)

# --- 5. Calcular porcentaje de aumento por periodo ---
periods = []
for i in range(len(change_points) - 1):
    start, end = change_points[i], change_points[i + 1]
    v_ini, v_fin = values[start], values[end]
    if v_ini == 0:
        pct = np.nan
    else:
        pct = ((v_fin - v_ini) / v_ini) * 100
    periods.append((years[start], years[end], pct))

# --- 6. Graficar ---
plt.figure(figsize=(10, 6))
plt.plot(years, values, 'o-', label='Datos originales', alpha=0.6)
plt.plot(years, smoothed, '-', color='blue', label='Serie suavizada', linewidth=2)

# Marcar puntos de cambio
for i in change_points[1:-1]:
    plt.axvline(years[i], color='red', linestyle='--', linewidth=1.5)
    plt.text(years[i], max(values)*0.9, str(years[i]), rotation=90,
             color="red", fontsize=9, ha="center", va="bottom")

# Etiquetas con % de crecimiento
for (start, end, pct) in periods:
    mid = (start + end) / 2
    plt.text(mid, max(values)*0.7, f"{start}-{end}\n{pct:.1f}%", 
             ha="center", fontsize=9, bbox=dict(facecolor="white", alpha=0.6))

plt.title("Segmentación de periodos mediante K-Means + suavizado gaussiano")
plt.xlabel("Año")
plt.ylabel("Número de publicaciones")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("periodos_kmeans_suavizado.png", dpi=300, bbox_inches="tight")
plt.show()

# --- 7. Mostrar resumen en consola ---
print("Periodos detectados con porcentaje de aumento:")
for i, (start, end, pct) in enumerate(periods, 1):
    print(f"Periodo {i}: {start}–{end} | Crecimiento = {pct:.2f}%")
